<a href="https://colab.research.google.com/github/acpotgieter/geog510/blob/Lab-10/510lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/geog-510/blob/main/book/labs/lab_10.ipynb)

## Overview

This lab introduces Google Earth Engine (GEE) for accessing and analyzing geospatial data in Python. You will explore diverse data types, including DEM, satellite imagery, and land cover datasets. You’ll gain experience creating cloud-free composites, visualizing temporal changes, and generating animations for time-series data.


## Objectives

By completing this lab, you will be able to:

1. Access and visualize Digital Elevation Model (DEM) data for a specific region.
2. Generate cloud-free composites from Sentinel-2 or Landsat imagery for a specified year.
3. Visualize National Agricultural Imagery Program (NAIP) data for U.S. counties.
4. Display watershed boundaries using Earth Engine styling.
5. Visualize land cover changes over time using split-panel maps.
6. Create a time-lapse animation for land cover changes over time in a region of your choice.


In [1]:
%pip install -U "geemap[workshop]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.7 MB/s eta 0:00:00


In [2]:
import ee
import geemap
geemap.ee_initialize(project='ee-acpotgieter')

## Exercise 1: Visualizing DEM Data

Find a DEM dataset in the [Earth Engine Data Catalog](https://developers.google.com/earth-engine/datasets) and clip it to a specific area (e.g., your country, state, or city). Display it with an appropriate color palette. For example, the sample map below shows the DEM of the state of Colorado.

![](https://i.imgur.com/OLeSt7n.png)

In [9]:
m = geemap.Map(center=[35.746512, -86.209818], zoom=8)

states = ee.FeatureCollection("TIGER/2018/States")
fc = states.filter(ee.Filter.eq("NAME", "Tennessee"))

image = ee.Image("USGS/SRTMGL1_003").clipToCollection(fc)
vis_params = {
    "min": 0,
    "max": 3500,
    "palette": "terrain",
}
m.addLayer(image, vis_params, "DEM")

m.addLayer(fc.style(**{"color": "ff0000ff", "fillColor": "00000000"}), {}, "Tennessee")
m.centerObject(fc)
m.add_text("Created by A.C. Potgieter")
m

Map(center=[35.746512, -86.209818], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

## Exercise 2: Cloud-Free Composite with Sentinel-2 or Landsat

Use Sentinel-2 or Landsat-9 data to create a cloud-free composite for a specific year in a region of your choice.

Use [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) or [Landsat-9 data](https://developers.google.com/earth-engine/datasets/catalog/landsat-9) data to create a cloud-free composite for a specific year in a region of your choice. Display the imagery on the map with a proper band combination. For example, the sample map below shows a cloud-free false-color composite of Sentinel-2 imagery of the year 2021 for the state of Colorado.

![](https://i.imgur.com/xkxpkS1.png)

In [7]:
m = geemap.Map(center=[35.746512, -86.209818], zoom=8)

states = ee.FeatureCollection("TIGER/2018/States")
Tennessee = states.filter(ee.Filter.eq("NAME", "Tennessee"))

collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
images = (
    collection.filterBounds(Tennessee)
    .filterDate("2023-01-01", "2023-12-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
)
images.size()

composite = images.median().clip(Tennessee)

vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B8", "B4", "B3"],
}

m.addLayer(composite, vis, "Sentinel-2 Composite")
m.centerObject(Tennessee, 8)
m

Map(center=[35.8579904338385, -86.35074162079951], controls=(WidgetControl(options=['position', 'transparent_b…

## Exercise 3: Visualizing NAIP Imagery

Use [NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ) imagery to create a cloud-free imagery for a U.S. county of your choice. For example, the sample map below shows a cloud-free true-color composite of NAIP imagery for Knox County, Tennessee. Keep in mind that there might be some counties with the same name in different states, so make sure to select the correct county for the selected state.

![](https://i.imgur.com/iZSGqGS.png)

In [8]:
counties = ee.FeatureCollection('TIGER/2018/Counties')

knox_tn = counties.filter(ee.Filter.And(
    ee.Filter.eq('NAME', 'Knox'),
    ee.Filter.eq('STATEFP', '47')
))

outline_vis_params = {
    'color': 'FF0000',
    'fillColor': '00000000',
    'width': 2
}

m = geemap.Map()

m.addLayer(knox_tn.style(**outline_vis_params), {}, 'Knox County Outline')

m.centerObject(knox_tn, 10)

naip = ee.ImageCollection('USDA/NAIP/DOQQ') \
    .filterBounds(knox_tn.geometry()) \
    .filterDate('2020-01-01', '2023-12-31')

cloud_free_naip = naip.median().clip(knox_tn.geometry())

naip_vis_params = {
    'bands': ['N', 'R', 'G'],
    'min': 0,
    'max': 255,
    'gamma': 1.4
}

m.addLayer(cloud_free_naip, naip_vis_params, 'NAIP Cloud-Free Composite')

m.add_text("Created by A.C. Potgieter")
m

Map(center=[35.993196068178555, -83.93723059189078], controls=(WidgetControl(options=['position', 'transparent…

## Exercise 4: Visualizing Watershed Boundaries

Visualize the [USGS Watershed Boundary Dataset](https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC04) with outline color only, no fill color.

![](https://i.imgur.com/PLlNFq3.png)

In [3]:
wbd = ee.FeatureCollection("USGS/WBD/2017/HUC06")

outline_style = {
    "color": "blue",
    "width": 2,
    "fillColor": "00000000"
}

Map = geemap.Map(center=[39.8283, -98.5795], zoom=4)
Map.addLayer(wbd.style(**outline_style), {}, "Watershed Boundaries")
Map.add_text("Created by A.C. Potgieter")
Map

Map(center=[39.8283, -98.5795], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## Exercise 5: Visualizing Land Cover Change

Use the [USGS National Land Cover Database](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2019_REL_NLCD) and [US Census States](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2018_States) to create a split-panel map for visualizing land cover change (2001-2019) for a US state of your choice. Make sure you add the NLCD legend to the map.

![](https://i.imgur.com/Au7Q5Ln.png)

In [4]:
from geemap.legends import builtin_legends

for legend in builtin_legends:
    print(legend)

states = ee.FeatureCollection("TIGER/2018/States")
state = states.filter(ee.Filter.eq("NAME", "Tennessee"))

split = geemap.Map(center=[39.3210, -111.0937], zoom=7, height=600)

nlcd_vis = {
    "min": 0,
    "max": 95,
    "palette": [
        "#476ba1", "#d1defa", "#decaca", "#d99482", "#ee0000", "#ab0000", "#b3aea3",
        "#68ab5f", "#1c5f2c", "#b5ca8f", "#af963c", "#ccb879", "#dfdfc2", "#d1d182",
        "#a3cc51", "#82ba9e", "#dcd939", "#ab6c28", "#b8d9eb", "#6c9fb8"
    ],
}

m = geemap.Map(center=[35.85, -86.35], zoom=8)

nlcd_2001 = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2001").select("landcover").clip(state)
nlcd_2019 = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2019").select("landcover").clip(state)

left_layer = geemap.ee_tile_layer(nlcd_2001, {}, "NLCD 2001")
right_layer = geemap.ee_tile_layer(nlcd_2019, {}, "NLCD 2019")

m.split_map(left_layer, right_layer)

m.add_legend(builtin_legend="NLCD", max_width="100px", height="455px")

m

NLCD
ESA_WorldCover
ESRI_LandCover
ESRI_LandCover_TS
Dynamic_World
NWI
MODIS/051/MCD12Q1
MODIS/006/MCD12Q1
GLOBCOVER
JAXA/PALSAR
Oxford
AAFC/ACI
COPERNICUS/CORINE/V20/100m
COPERNICUS/Landcover/100m/Proba-V/Global
USDA/NASS/CDL
ALOS_landforms


Map(center=[35.85, -86.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Exercise 6: Creating a Landsat Timelapse Animation

Generate a timelapse animation using Landsat data to show changes over time for a selected region.

![Spain](https://github.com/user-attachments/assets/f12839c0-1c30-404d-b0ab-0fa12ce12d24)

In [6]:
m = geemap.Map()
roi = ee.Geometry.BBox(113.8252, 22.1988, 114.0851, 22.3497)
m.add_layer(roi)
m.center_object(roi)
m

timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="hong_kong.gif",
    start_year=1990,
    end_year=2024,
    start_date="01-01",
    end_date="12-31",
    bands=["SWIR1", "NIR", "Red"],
    frames_per_second=2,
    title="Hong Kong",
)
geemap.show_image(timelapse)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/hong_kong.gif


Output()